In [42]:
#!pip freeze > requirements.txt

In [1]:
#Musical Machine Orchestration
#By Gissel Velarde
#Update 18.6.2025
#28.5.2025
import numpy as np
import os
import pandas as pd
import sys
sys.path.append('amos') 
from midi2df2midi import midi_to_dataframe, save_midi_from_df

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
import time 

In [6]:
"""
Code generated with https://platform.openai.com/
Model gpt-4.1, text.format: text, temp: 1.00, tokens: 2048, top_p: 1.00, store: true
Prompt by Gissel Velarde
Prompt:
the first 4 columns of a an numpy array with 8 columns always appear in quaterna. Write a function that learns the quaterna, given that column 3 is used as a label for a machine learning model. Then, once the model predicts the label for column 4, fill the corresponding values for columns 0, 1 and 2.
"""
#import numpy as np

def learn_quaterna_mapping(array):
    """
    Learns the mapping from label in column 3 to columns 0, 1, 2.
    Returns a dictionary: label_val -> [col0, col1, col2]
    """
    mapping = {}
    for row in array:
        label = row[3]
        # Only keep the last if duplications appear (can add check for consistency)
        mapping[label] = row[:3].tolist()
    return mapping

In [7]:
"""
Code generated with https://platform.openai.com/
Model gpt-4.1, text.format: text, temp: 1.00, tokens: 2048, top_p: 1.00, store: true
Prompt by Gissel Velarde
Prompt:
the first 4 columns of a an numpy array with 8 columns always appear in quaterna. Write a function that learns the quaterna, given that column 3 is used as a label for a machine learning model. Then, once the model predicts the label for column 4, fill the corresponding values for columns 0, 1 and 2.
"""
def fill_quaterna_columns(predicted_labels, quaterna_mapping):
    """
    Given a list/array of predicted labels, use the mapping to reconstruct cols 0, 1, 2.
    Returns an array of shape (N, 3) where N is the number of predicted labels.
    """
    filled = []
    for label in predicted_labels:
        if label in quaterna_mapping:
            filled.append(quaterna_mapping[label])
        else:
            # handle unknown labels (e.g., with np.nan)
            filled.append([np.nan, np.nan, np.nan])
    return np.array(filled)

In [38]:
#By Gissel Velarde
#Update 18.6.2025
#28.5.2025
def split_and_encode(X, y, test_size=0.2, random_state=42):
    if test_size>0:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)
    elif test_size==0:
        X_train = X
        y_train = y
        X_test, y_test  = 0, 0 #We will no use X_test, y_test for inference
    le = LabelEncoder()
    le.fit(y_train)  # Fit only on train 
    print("y_train, test size:",test_size,", labels:", np.unique(y_train))
    y_train = le.transform(y_train)  # Will be 0,1,...,N-1
    if test_size>0:
        y_test = le.transform(y_test)
    return X_train, X_test, y_train, y_test, le

In [39]:
#By Gissel Velarde
#Update 18.6.2025
#28.5.2025
def clf_predict(X2,le,model,mapping):
    y_pred = model.predict(X2)
    print("Predictions ",np.unique(y_pred))
    # inverse transform to obtained the original labels:
    y_pred_orig = le.inverse_transform(y_pred)
    # Fill columns 0, 1, 2 using the mapping
    new_cols = fill_quaterna_columns(y_pred_orig, mapping)
    print("Predictions map",np.unique(y_pred_orig))
    nmat = np.concatenate((new_cols, y_pred_orig.reshape(-1, 1),X2), axis=1)
    return nmat

In [40]:
# By G. Velarde from
#16.6.2025
def ml_exp(filein, fileout):
    #Adapted from:
    # https://scikit-learn.org/stable/auto_examples/classification/plot_classifier_comparison.html
    # Authors: The scikit-learn developers
    # SPDX-License-Identifier: BSD-3-Clause
    names = [
        "Nearest_Neighbors",
        "Linear_SVM",
        "Decision_Tree",
        "Random_Forest",
        "Neural_Net",
        "AdaBoost",
        "Naive_Bayes",
        "XGBoost",
    ]

    classifiers = [
        KNeighborsClassifier(1),
        SVC(kernel="linear"),
        DecisionTreeClassifier(),
        RandomForestClassifier(),
        MLPClassifier(),
        AdaBoostClassifier(),
        GaussianNB(),
        XGBClassifier(),
    ]
    # midi to dataframe
    dfnmat = midi_to_dataframe(filein)
    # sort by onset, duration, track number
    dfnmat = dfnmat.sort_values(
        ['onset in quarter notes','duration in quarter notes', 'track number'],
        ascending=[True, True, True]
    )
    # convert df to nmat array
    nmat = dfnmat.to_numpy()  
    # Learn mapping from col3 /programs to cols 0-2, track, track name, channel
    mapping = learn_quaterna_mapping(nmat)
    print("mapping", mapping)
    X = nmat[:, 4:8]  # onset, duration, pitch, velocity
    y = nmat[:, 3]    # tracks
    print("Labels", np.unique(y))
    print("Number of events in ",filein,":", X.shape[0])
    print("last onset at ", X[X.shape[0]-1, 0])
    rng = np.random.RandomState(2)
    ### The outfile
    dfnmat = midi_to_dataframe(fileout) 
    #sort by onset, duration, track number
    dfnmat=dfnmat.sort_values(['onset in quarter notes','duration in quarter notes', 'track number'],ascending=[True, True, True])
    #convert df to nmat array
    nmat2=dfnmat.to_numpy()
    X2 = nmat2[:,4:8] #onset, duration, pitch, velocity
    print("Number of events in",fileout,":",X2.shape[0])
    print("last onset at ",X2[X2.shape[0]-1,0])
    #Partion the dataset
    X_train, X_test, y_train, y_test,le = split_and_encode(X, y, test_size=0.2, random_state=42) #For evaluation
    X_train_f, X_test_f, y_train_f, y_test_f,le_f = split_and_encode(X, y, test_size=0, random_state=42) #For orchestration
    # iterate over classifiers
    for name, clf in zip(names, classifiers):
        start = time.time()
        clf.fit(X_train, y_train)
        score = clf.score(X_test, y_test)
        end = time.time()
        print("---------",name)
        print("Train Time (sec) :",f"{end - start:.4f}")
        print("Score on Test (20%): ", f"{score:.4f}") 
        #_= ConfusionMatrixDisplay.from_estimator(clf, X_test, y_test_encoded)
        #Predict orchestration 
        #Retrain with the full input file
        clf.fit(X_train_f, y_train_f)
        data=clf_predict(X2,le_f,clf,mapping)
        #convert nmat array to df
        #https://www.geeksforgeeks.org/convert-numpy-array-to-dataframe/
        #Specifying Column Names 				pitch	velocity
        dfdata = pd.DataFrame(data, columns=['track number','track name', 'channel', 'program','onset in quarter notes','duration in quarter notes','pitch','velocity'])
        #convert track name to string
        dfdata['track name'] = dfdata['track name'].astype(str)
        extensions = name + ".mid"
        filename = fileout.replace(".mid", extensions)
        print('Orchestration:',filename)
        save_midi_from_df(dfdata, filename)

In [41]:
#Usage
#ml_exp(filein,fileout)
#filein corresponds to the file used to learn the orchestration style and instrument map
#fileout is the file used to transfer the learned orchestration style.
#ml_exp will save a new MIDI files for each ML model
ml_exp('midis/sugar-plum-fairy_orch.mid','midis/fur-elise.mid')

mapping {45: [13, 'Contrabass', 1], 8: [8, 'Celesta', 0], 71: [4, '2 Clarinets in A', 7], 69: [3, 'English Horn', 5], 70: [6, '2 Bassoons', 10], 73: [1, '3 Flutes', 3], 68: [2, '2 Oboes', 4], 60: [7, '4 Horns in F', 12], 48: [9, 'Violin I', 13]}
Labels [8 45 48 60 68 69 70 71 73]
Number of events in  midis/sugar-plum-fairy_orch.mid : 1824
last onset at  103.5
Number of events in midis/fur-elise.mid : 1020
last onset at  186.5
y_train, test size: 0.2 , labels: [8 45 48 60 68 69 70 71 73]
y_train, test size: 0 , labels: [8 45 48 60 68 69 70 71 73]
--------- Nearest_Neighbors
Train Time (sec) : 0.0055
Score on Test (20%):  0.8877
Predictions  [1 2 3 6 7 8]
Predictions map [45 48 60 70 71 73]
Orchestration: midis/fur-eliseNearest_Neighbors.mid
--------- Linear_SVM
Train Time (sec) : 4.3959
Score on Test (20%):  0.8027
Predictions  [0 1 2]
Predictions map [8 45 48]
Orchestration: midis/fur-eliseLinear_SVM.mid
--------- Decision_Tree
Train Time (sec) : 0.0021
Score on Test (20%):  0.9178
Pre